#### 【分析需求】

- **多維度行為追蹤**：在不減少資料列數的情況下，同步計算單一客戶的「首購」、「前次交易」與「近期移動趨勢」。
    
- **業績貢獻透明化**：計算單筆訂單對於全公司總體營收的佔比，並透過金額門檻進行訂單分級。
    
- **平滑營收波動**：透過移動平均消除單次大額或小額訂單的雜訊，以觀察客戶的真實消費力道。
    

#### 【結果解讀與商業價值】

- #### 客戶成長路徑：
    

1. 首購與回購對比：若「差額」長期為正值，代表客戶黏著度高且消費力持續升級。
2. 移動平均價值：若移動平均持續下滑，即便當前訂單標籤為「高額」，仍需警惕該客戶是否有流失風險。

- **標籤管理：**

1. **高額（\>= 25萬）**<span style="color: var(--vscode-foreground);">：定義為 VIP 訂單，應觸發優先配送服務。</span>
2. **中額（15萬-25萬）**：定義為主力推廣對象，具備升級為高額客戶的潛力。
3. **低額（\< 15萬）：定義為基礎客層，應降低行銷資源以維護成本。**

- **業績權重**：百分比能幫助管理者一眼識別出「撐起公司營收」的關鍵交易，對於風險控管（避免營收過於集中在特定客戶）極具參考價值。

In [3]:
SELECT Date,
       Client,
       Amount,
       FIRST_VALUE(Amount) OVER(PARTITION by client ORDER by Date) as [首購金額],
       LAG(Amount,1,0) OVER(PARTITION by client ORDER by Date) as [上一筆訂單金額],
       Amount-LAG(Amount,1,0) OVER(PARTITION by client ORDER by Date) as [差額],
       CONCAT(round(Amount*100/SUM(Amount) OVER(),2),'%') as [業績百分比],
       AVG(Amount) OVER(PARTITION by client ORDER by Date ROWS between 2 preceding and current ROW) [移動平均],
       case
       when Amount>=250000 then '高額'
       when Amount>=150000 then '中額'
       else '低額'
        end as [訂單分級]
  FROM HappyFruit.dbo.salesall;

(1000 rows affected)

Total execution time: 00:00:00.114

Date,Client,Amount,首購金額,上一筆訂單金額,差額,業績百分比,移動平均,訂單分級
2019-01-04,69小華超商,154000.00,154000.00,0.00,154000.00,0.09%,154000.00,中額
2019-01-07,69小華超商,93000.00,154000.00,154000.00,-61000.00,0.06%,123500.00,低額
2019-01-09,69小華超商,247000.00,154000.00,93000.00,154000.00,0.15%,164666.6666,中額
2019-01-21,69小華超商,199000.00,154000.00,247000.00,-48000.00,0.12%,179666.6666,中額
2019-01-25,69小華超商,255000.00,154000.00,199000.00,56000.00,0.15%,233666.6666,高額
2019-02-02,69小華超商,130000.00,154000.00,255000.00,-125000.00,0.08%,194666.6666,低額
2019-02-03,69小華超商,296000.00,154000.00,130000.00,166000.00,0.18%,227000.00,高額
2019-02-08,69小華超商,228000.00,154000.00,296000.00,-68000.00,0.14%,218000.00,中額
2019-02-10,69小華超商,98000.00,154000.00,228000.00,-130000.00,0.06%,207333.3333,低額
2019-02-19,69小華超商,239000.00,154000.00,98000.00,141000.00,0.14%,188333.3333,中額
